# Práctica 1 · Paso 3 — Curva de gasto, serie de caudales y curva de duración de tu cuenca
### Hidrología (1CIA-502) · Ingeniería Ambiental · ESPAM MFL · Unidad I
**Docente:** Ing. Cristian Paliz Acosta, MSc.

Trabajas con **datos reales** de la estación hidrológica del INAMHI que representa a tu cuenca y con la
simulación de GEOGloWS corregida con esos datos.

| Bloque | Pregunta que responde | Producto |
|---|---|---|
| 3 | ¿Qué estación representa a mi cuenca? | Ficha de la estación |
| 4 | ¿Cómo se pasa del nivel que se lee al caudal? | **Curva de gasto** Q = a (h − h₀)ᵇ |
| 5 | ¿Cómo se comporta el río día a día y mes a mes? | Hidrograma y régimen observados |
| 6 | ¿Cuánto me puedo fiar del modelo global? | Observado vs. GEOGloWS sin corregir y corregido |
| 7 | ¿Cuánta agua pasa en la salida de **mi** cuenca? | Serie diaria 1995–2014 en tu punto de salida |
| 8 | ¿Con qué caudal puedo contar? | **Curva de duración**, Q10, Q50, Q90, Q95 |

**Mano antes que modelo:** antes de la sección 4 escribe en la celda 1 tu estimación de h₀ mirando la
gráfica de los pares nivel–caudal.

`Archivo → Guardar una copia en Drive` y ejecuta en orden con `Shift + Enter`. Solo se edita la **celda 1**.

## 1 · CONFIGURACIÓN ✍️ (única celda que editas)
- **CUENCA:** la misma de los pasos 1 y 2.
- **ESTACION:** `None` usa la estación de referencia de tu cuenca (tabla de la sección 3). Puedes poner otra, p. ej. `"H272"`.
- **EXUTORIO:** las coordenadas UTM 17S de **tu** punto de salida del paso 2; `None` = salida de la cuenca completa.
- **AREA_PASO2_KM2:** el área que obtuviste en el paso 2 (para el expediente y la comparación).
- **H0_MANO:** tu estimación de h₀, en metros, antes de ejecutar la sección 4.
- **AFOROS_PROPIOS:** `True` si tienes pares nivel–caudal propios (CSV con `fecha, nivel_m, caudal_m3s`).

In [ ]:
ESTUDIANTE = "Apellidos Nombres"
CUENCA     = "Carrizal"      # Ayampe · Carrizal · Chone · Jama · Jipijapa · Portoviejo

ESTACION   = None            # None = estación de referencia de tu cuenca
EXUTORIO   = None            # (Este, Norte) en UTM 17S, el mismo del paso 2; None = salida de la cuenca
AREA_PASO2_KM2 = None        # el área de tu cuenca o subcuenca según el paso 2

H0_MANO = None               # tu estimación de h0 en metros, p. ej. 0.15
AFOROS_PROPIOS = False

PERIODO = (1995, 2014)       # periodo de referencia del protocolo común
DATOS = "https://raw.githubusercontent.com/crispaliz/datos-manabi/main/caudales"

## 2 · Cargar los datos de caudal de Manabí
Observados del INAMHI (anuarios 1993–2013), simulación de GEOGloWS (1980–2023) y su versión corregida. Tarda unos segundos.

In [ ]:
import os, io, unicodedata, numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
SLUG = unicodedata.normalize("NFKD", ESTUDIANTE).encode("ascii", "ignore").decode().replace(" ", "_")
os.makedirs("salidas", exist_ok=True)
leer = lambda f, **k: pd.read_csv(f"{DATOS}/{f}", **k)
EST  = leer("estaciones_hidrologicas.csv")
SAL  = leer("salidas_cuencas.csv").set_index("cuenca")
TRA  = leer("tramos_geoglows.csv")
CG   = leer("curvas_gasto.csv")
MET  = leer("metricas_correccion.csv").set_index("estacion_id")
PAR  = leer("pares_nivel_caudal_1993_2013.csv", parse_dates=["fecha"])
OBS  = leer("observados/manabi_series_1993_2013.csv", parse_dates=["fecha"])
SIM  = leer("geoglows_simulado_1980_2023.csv", parse_dates=["fecha"], index_col="fecha")
COR  = leer("geoglows_corregido_estaciones_1980_2023.csv", parse_dates=["fecha"], index_col="fecha")
OBS["estacion_id"] = OBS.codigo.replace({"HA2D": "H1081", "HA48": "H1121"})
OBS = OBS[OBS.estado_mes != "discrepa"]
print(f"✅ {len(EST)} estaciones · {SIM.shape[1]} tramos simulados · {len(OBS):,} valores diarios observados".replace(",", " "))

## 3 · La estación que representa a tu cuenca
Una estación hidrológica mide el nivel del río (limnímetro o limnígrafo) y, con la curva de gasto, lo convierte
en caudal. La **calidad de la corrección** dice cuánto se parece GEOGloWS corregido a lo observado en esa estación
(validación cruzada: KGE y NSE mensual).

Si tu cuenca no tiene estación con datos suficientes, el cuaderno usa una vecina y **lleva** su caudal a tu
cuenca con la relación que da el modelo (sección 7). Eso agrega incertidumbre: decláralo en tu capítulo.

In [ ]:
POR_DEFECTO = {"Carrizal": "H229", "Portoviejo": "H1081", "Chone": "H229",
               "Jama": "H206", "Jipijapa": "H272", "Ayampe": "H267"}
EID = ESTACION or (SAL.loc[CUENCA, "estacion_referencia"] if isinstance(SAL.loc[CUENCA, "estacion_referencia"], str)
                   else POR_DEFECTO[CUENCA])
e = EST.set_index("estacion_id").loc[EID]
propia = e.cuenca == CUENCA
print("Estaciones hidrológicas dentro de tu cuenca:")
display(EST[EST.cuenca == CUENCA][["estacion_id", "estacion", "dias_caudal", "dias_nivel", "calidad_correccion"]])
print(f"\nEstación usada: {EID} {e.estacion} · cuenca {e.cuenca or '—'} · {e.dias_caudal} días con caudal · "
      f"corrección {e.calidad_correccion}")
if not propia:
    print(f"⚠️  {EID} no está en la cuenca {CUENCA}: su caudal se llevará a tu cuenca con la relación del modelo (sección 7).")
CORREGIDA = EID in COR.columns
if not CORREGIDA:
    print("⚠️  La corrección de GEOGloWS en esta estación no es confiable: se usan solo los años observados.")

# Tramo de salida: el de la cuenca o el más cercano a tu exutorio
if EXUTORIO is None:
    comid_sal = int(SAL.loc[CUENCA, "comid_salida"])
else:
    from pyproj import Transformer
    lon, lat = Transformer.from_crs(32717, 4326, always_xy=True).transform(*EXUTORIO)
    cand = TRA[TRA.cuenca == CUENCA].copy()
    cand["d"] = np.hypot((cand.longitud - lon) * 111.3, (cand.latitud - lat) * 110.6)
    comid_sal = int(cand.sort_values("d").comid.iloc[0])
    print(f"Tu exutorio queda a {cand.d.min():.1f}".replace(".", ",") + f" km del tramo de GEOGloWS {comid_sal}.")
    if cand.d.min() > 3:
        print("⚠️  La red de GEOGloWS es gruesa: el tramo más cercano puede drenar un área distinta de tu subcuenca. "
              "Compara su caudal con tu área (caudal específico, sección 9) y decláralo.")
t = TRA.set_index("comid").loc[comid_sal]
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(TRA.longitud, TRA.latitud, s=10, c="#C9D6DE", label="Tramos GEOGloWS")
cc = TRA[TRA.cuenca == CUENCA]; ax.scatter(cc.longitud, cc.latitud, s=16, c="#1F6F8B", label=f"Tramos de {CUENCA}")
ax.scatter(EST.longitud, EST.latitud, marker="^", s=40, c="#12263A", label="Estaciones INAMHI")
ax.scatter([e.longitud], [e.latitud], marker="^", s=160, c="#B3541A", ec="k", label=f"Estación usada {EID}")
ax.scatter([t.longitud], [t.latitud], marker="*", s=260, c="crimson", ec="k", label="Salida de tu cuenca")
ax.legend(fontsize=8, loc="lower left"); ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title(f"Estación y salida · {CUENCA}", fontweight="bold"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("salidas/P3_00_estacion.png", dpi=150); plt.show()

## 4 · Curva de gasto: del nivel al caudal
El observador lee el **nivel** h en la regla. El caudal no se mide todos los días: se mide unas veces al año
(aforos) y con esos pares se ajusta la curva **Q = a (h − h₀)ᵇ**:

- **h₀** es el nivel en el que el caudal es cero (la regla no empieza en el fondo del cauce).
- **b** depende de la forma de la sección: cerca de 1,5 en cauces rectangulares anchos; 2 a 3 en cauces en V.
- Con log Q = log a + b · log (h − h₀), la curva es una **recta** en escala logarítmica: se prueba una serie de
  h₀ y se queda el que da la mejor recta (mayor R²).

⚠️ Los pares de esta sección son **nivel y caudal del mismo día publicados en el anuario**: el caudal ya salió de
la curva oficial del INAMHI. Por eso caen casi sobre una línea. Si la nube se abre en dos ramas, **la curva cambió**
entre años: el cauce se socavó o se sedimentó.

In [ ]:
if AFOROS_PROPIOS:
    from google.colab import files
    print("Sube tu CSV de aforos (fecha, nivel_m, caudal_m3s):"); up = files.upload()
    pr = pd.read_csv(io.BytesIO(list(up.values())[0]), parse_dates=["fecha"])
else:
    pr = PAR[PAR.estacion_id == EID].copy()
if len(pr) < 30:
    raise SystemExit(f"❌ {EID} tiene muy pocos pares nivel–caudal. Elige otra estación en la celda 1.")
anios = sorted(pr.fecha.dt.year.unique())
ult = pr[pr.fecha.dt.year.isin(anios[-3:])]
fig, ax = plt.subplots(figsize=(7.5, 5))
sc = ax.scatter(pr.nivel_m, pr.caudal_m3s, c=pr.fecha.dt.year, cmap="viridis", s=6)
plt.colorbar(sc, label="Año"); ax.set_xlabel("Nivel h (m)"); ax.set_ylabel("Caudal Q (m³/s)")
ax.set_title(f"Pares nivel–caudal · {EID} {e.estacion}", fontweight="bold"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"{len(pr)} pares de {anios[0]} a {anios[-1]}. Se ajusta con los últimos tres años ({anios[-3:][0]}–{anios[-1]}): "
      f"{len(ult)} pares, niveles de {ult.nivel_m.min():.2f} a {ult.nivel_m.max():.2f} m.")
if H0_MANO is None:
    print("✍️  Escribe tu estimación de h0 en la celda 1 (H0_MANO) antes de seguir.")

In [ ]:
h, q = ult.nivel_m.values, ult.caudal_m3s.values
filas = []
for h0 in np.linspace(h.min() - 1.0, h.min() - 0.005, 400):   # h0 por debajo del nivel más bajo leído, hasta 1 m
    x, y = np.log(h - h0), np.log(q); b, la = np.polyfit(x, y, 1)
    filas.append((h0, np.exp(la), b, np.corrcoef(x, y)[0, 1] ** 2))
busq = pd.DataFrame(filas, columns=["h0", "a", "b", "R2"])
mejor = busq.loc[busq.R2.idxmax()]; H0, A, B, R2 = mejor.h0, mejor.a, mejor.b, mejor.R2
Qc = lambda hh: A * np.clip(hh - H0, 0, None) ** B
fmt_curva = lambda a, h0, b: f"Q = {a:.3f} (h {'−' if h0 >= 0 else '+'} {abs(h0):.3f})^{b:.3f}".replace(".", ",")
h_max_aj = h.max()
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].plot(busq.h0, busq.R2, color="#1F6F8B"); axs[0].axvline(H0, color="#B3541A", ls="--", label=f"h₀ del ajuste = {H0:.2f} m")
if H0_MANO is not None: axs[0].axvline(H0_MANO, color="k", ls=":", label=f"tu h₀ a mano = {H0_MANO:.2f} m")
axs[0].set_xlabel("h₀ probado (m)"); axs[0].set_ylabel("R² de la recta en log–log"); axs[0].legend(); axs[0].grid(alpha=0.3)
axs[0].set_title("Búsqueda de h₀", fontweight="bold")
hh = np.linspace(max(H0, 0), max(pr.nivel_m.max(), h_max_aj) * 1.05, 200)
axs[1].scatter(pr.nivel_m, pr.caudal_m3s, s=5, c="#C9D6DE", label="Todos los pares")
axs[1].scatter(h, q, s=6, c="#12263A", label="Pares del ajuste")
axs[1].plot(hh, Qc(hh), color="#B3541A", lw=2.5, label=fmt_curva(A, H0, B))
axs[1].axvspan(h_max_aj, hh.max(), color="#F2A65A", alpha=0.15, label="Extrapolación")
axs[1].set_xlabel("Nivel h (m)"); axs[1].set_ylabel("Caudal Q (m³/s)"); axs[1].legend(fontsize=9); axs[1].grid(alpha=0.3)
axs[1].set_title(f"Curva de gasto · R² = {R2:.4f}", fontweight="bold")
plt.tight_layout(); plt.savefig("salidas/P3_01_curva_gasto.png", dpi=150); plt.show()
print(fmt_curva(A, H0, B), f"  R² = {R2:.4f}".replace(".", ","))
for _, r in CG[CG.estacion_id == EID].iterrows():
    print(f"   referencia {r.periodo.replace('_', ' '):16s} ({r.anios}):", fmt_curva(r.a, r.h0_m, r.b), f" R² = {r.R2:.3f}".replace(".", ","))
print("Si la curva del periodo completo tiene un R² bastante menor, la sección del río cambió entre años.")

## 5 · La serie observada: hidrograma y régimen
Los huecos son días sin dato en los anuarios. El caudal **medio** y la **mediana** (Q50) no son lo mismo:
en los ríos de la Costa unas pocas crecidas suben mucho la media.

In [ ]:
qo = OBS[(OBS.estacion_id == EID) & (OBS.variable == "caudal")].drop_duplicates("fecha").set_index("fecha").valor.astype(float)
qo = qo.reindex(pd.date_range(qo.index.min(), qo.index.max()))
nv = OBS[(OBS.estacion_id == EID) & (OBS.variable == "nivel")].drop_duplicates("fecha").set_index("fecha").valor / 100
pct_dias_ext = 100 * (nv > h_max_aj).mean()
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [2.3, 1]})
axs[0].plot(qo.index, qo.values, color="#1F6F8B", lw=0.7)
axs[0].axhline(qo.mean(), color="#B3541A", ls="--", label=f"media {qo.mean():.2f} m³/s")
axs[0].axhline(qo.median(), color="#12263A", ls=":", label=f"mediana {qo.median():.2f} m³/s")
axs[0].set_ylabel("Caudal diario (m³/s)"); axs[0].legend(frameon=False); axs[0].grid(alpha=0.3)
axs[0].set_title(f"Caudal observado · {EID} {e.estacion} · {qo.notna().mean()*100:.0f} % de días con dato", fontweight="bold")
mens = qo.groupby(qo.index.month).mean()
axs[1].bar(mens.index, mens.values, color="#1F6F8B"); axs[1].set_xticks(range(1, 13)); axs[1].set_xticklabels(list("EFMAMJJASOND"))
axs[1].set_ylabel("Caudal medio mensual (m³/s)"); axs[1].set_title("Régimen mensual", fontweight="bold"); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("salidas/P3_02_hidrograma_observado.png", dpi=150); plt.show()
print(f"Días con nivel por encima del rango de la curva ajustada ({h_max_aj:.2f} m): {pct_dias_ext:.1f} %")

## 6 · ¿Cuánto me puedo fiar del modelo global?
GEOGloWS simula el caudal de todos los ríos del mundo con lluvia de reanálisis. No conoce las represas ni la
curva de cada estación, así que **se corrige**: para cada mes, el caudal simulado se cambia por el caudal
observado que tiene la misma probabilidad de ser superado (mapeo por cuantiles).

Compara las tres curvas de duración en el mismo periodo. **KGE** y **NSE** valen 1 si el modelo es perfecto;
un NSE menor que 0 significa que la media observada predice mejor que el modelo.

In [ ]:
comid_est = int(e.comid_geoglows)
ix = qo.dropna().index.intersection(SIM.index)
dur = lambda s: (np.arange(1, len(s) + 1) / (len(s) + 1) * 100, np.sort(s.dropna().values)[::-1])
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
for s, lab, col, ls in [(qo[ix], "Observado", "#12263A", "-"), (SIM.loc[ix, str(comid_est)], "GEOGloWS sin corregir", "#F2A65A", "--")] + \
        ([(COR.loc[ix, EID], "GEOGloWS corregido", "#1F6F8B", "-")] if CORREGIDA else []):
    p, v = dur(s); axs[0].semilogy(p, np.clip(v, 1e-3, None), color=col, ls=ls, lw=2, label=lab)
    axs[1].plot(range(1, 13), s.groupby(s.index.month).mean().values, color=col, ls=ls, lw=2, marker="o", label=lab)
axs[0].set_xlabel("% del tiempo en que se iguala o supera"); axs[0].set_ylabel("Caudal (m³/s, log)")
axs[0].set_title(f"Curvas de duración · {EID} · días con dato", fontweight="bold"); axs[0].legend(); axs[0].grid(alpha=0.3, which="both")
axs[1].set_xticks(range(1, 13)); axs[1].set_xticklabels(list("EFMAMJJASOND")); axs[1].set_ylabel("Caudal medio mensual (m³/s)")
axs[1].set_title("Régimen mensual", fontweight="bold"); axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("salidas/P3_03_modelo_vs_observado.png", dpi=150); plt.show()
if EID in MET.index:
    m = MET.loc[EID]
    display(pd.DataFrame({"Sin corregir": [m.sesgo_pct_sin_corregir, m.KGE_sin_corregir, m.NSE_mensual_sin_corregir],
                          "Corregido (validación cruzada)": [m.sesgo_pct_validacion, m.KGE_validacion, m.NSE_mensual_validacion]},
                         index=["Sesgo del volumen (%)", "KGE diario", "NSE mensual"]).round(2))

## 7 · Llevar el caudal a la salida de tu cuenca
La estación casi nunca está justo en tu punto de salida. El modelo sí tiene un tramo en tu salida, así que se
usa como **regla de proporción**, mes a mes:

  Q salida (día) = Q estación corregido (día) × [Q simulado salida ÷ Q simulado estación] del mismo mes

Si la estación está en tu salida, la proporción es 1. Si es de otra cuenca, la proporción la pone el modelo:
la incertidumbre crece.

In [ ]:
s_sal, s_est = SIM[str(comid_sal)], SIM[str(comid_est)]
razon = (s_sal.groupby(s_sal.index.month).mean() / s_est.groupby(s_est.index.month).mean()).replace([np.inf], np.nan)
base = COR[EID] if CORREGIDA else qo.reindex(SIM.index)
Qs = (base * base.index.month.map(razon).values).rename("caudal_m3s")
Qp = Qs[(Qs.index.year >= PERIODO[0]) & (Qs.index.year <= PERIODO[1])].dropna()
print(f"Razón salida/estación (media anual): {np.average(razon, weights=s_est.groupby(s_est.index.month).mean()):.2f}")
print(f"Serie en tu salida, {PERIODO[0]}–{PERIODO[1]}: {len(Qp)} días con dato")
if len(Qp) < 3 * 365:
    print("⚠️  Menos de tres años: la curva de duración será poco representativa.")
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(Qp.index, Qp.values, color="#1F6F8B", lw=0.6)
ax.axhline(Qp.mean(), color="#B3541A", ls="--", label=f"media {Qp.mean():.2f} m³/s"); ax.legend(frameon=False)
ax.set_ylabel("Caudal diario (m³/s)"); ax.grid(alpha=0.3)
ax.set_title(f"Caudal diario en la salida de {CUENCA} · {PERIODO[0]}–{PERIODO[1]}", fontweight="bold")
plt.tight_layout(); plt.savefig("salidas/P3_04_serie_salida.png", dpi=150); plt.show()

## 8 · Curva de duración y caudales característicos de tu cuenca
Se ordenan los caudales de mayor a menor y a cada puesto *m* se le asigna p = m ÷ (n + 1) × 100 (Weibull).
**Qp** es el caudal igualado o superado el p % del tiempo.

| Caudal | Lectura |
|---|---|
| Q10 | Caudal alto: solo se supera el 10 % de los días |
| Q50 | Caudal mediano: la mitad de los días hay más, la mitad menos |
| Q90 | Caudal de estiaje con el que se puede contar 9 de cada 10 días |
| Q95 | Referencia habitual para el caudal ecológico |
| Q90 ÷ Q50 | Cerca de 1: río regular. Cerca de 0: río que casi se seca |

In [ ]:
qs = np.sort(Qp.values)[::-1]; n = len(qs); p = np.arange(1, n + 1) / (n + 1) * 100
car = {f"Q{k}": float(np.interp(k, p, qs)) for k in (10, 50, 90, 95)}
Qmed = Qp.mean(); pct_media = 100 * (Qp >= Qmed).mean()
po, vo = dur(qo.dropna())
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.semilogy(p, np.clip(qs, 1e-3, None), color="#1F6F8B", lw=3, label=f"Salida de {CUENCA}, {PERIODO[0]}–{PERIODO[1]}")
ax.semilogy(po, np.clip(vo, 1e-3, None), color="#12263A", lw=1.2, ls=":", label=f"Observado en {EID} (días con dato)")
for k, v in car.items():
    pp = int(k[1:]); ax.plot([pp, pp], [max(qs.min(), 1e-3), v], color="#B3541A", ls="--", lw=1); ax.plot(pp, v, "o", color="#B3541A")
    ax.text(pp + 1, v * (0.75 if k == "Q95" else 1.15), f"{k} = {v:.2f}", color="#B3541A")
ax.axhline(Qmed, color="k", lw=0.8, ls="-."); ax.text(55, Qmed * 1.12, f"media {Qmed:.2f} (se supera el {pct_media:.0f} % del tiempo)", fontsize=9)
ax.set_xlabel("% del tiempo en que el caudal es igualado o superado"); ax.set_ylabel("Caudal (m³/s, escala log)")
ax.set_title(f"Curva de duración · {CUENCA}", fontweight="bold"); ax.set_xlim(0, 100); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.savefig("salidas/P3_05_curva_duracion.png", dpi=150); plt.show()
print({k: round(v, 3) for k, v in car.items()}, "· Q90/Q50 =", round(car["Q90"] / car["Q50"], 2))

## 9 · Expediente y descarga
Guarda en el formato común lo que pasa a la ficha de síntesis y al proyecto integrador.

In [ ]:
fuente_txt = (f"{EID} {e.estacion}: observados INAMHI 1993–2013" +
              (f"; GEOGloWS corregido ({e.calidad_correccion})" if CORREGIDA else "; sin extensión con GEOGloWS") +
              ("" if propia else f"; llevado desde otra cuenca"))
f = f"P1_paso3 · {EID}"
exp = [("1.6", "Fuente de los datos de caudal", fuente_txt, "", "celda 3"),
       ("1.6", "Estación hidrológica", f"{EID} {e.estacion}", "", "celda 3"),
       ("1.6", "Calidad de la corrección", e.calidad_correccion, "", "metricas_correccion.csv"),
       ("1.6", "Tramo GEOGloWS de la salida", comid_sal, "comid", "celda 3"),
       ("1.6", "Periodo de la serie", f"{Qp.index.min():%Y-%m-%d} a {Qp.index.max():%Y-%m-%d}", "", f),
       ("1.6", "Número de pares nivel–caudal", len(ult), "n.º", f),
       ("1.6", "Curva de gasto a", round(A, 3), "—", f), ("1.6", "Curva de gasto h0", round(H0, 3), "m", f),
       ("1.6", "Curva de gasto b", round(B, 3), "—", f), ("1.6", "R² del ajuste", round(R2, 4), "—", f),
       ("1.6", "h0 estimado a mano", H0_MANO if H0_MANO is not None else "", "m", "celda 1"),
       ("1.6", "Nivel máximo del ajuste", round(h_max_aj, 2), "m", f),
       ("1.6", "Días por encima del rango de la curva", round(pct_dias_ext, 1), "%", f),
       ("1.6", "Q50 observado en la estación", round(qo.median(), 3), "m³/s", f),
       ("1.6", "Caudal medio", round(Qmed, 3), "m³/s", f),
       ("1.6", "Tiempo en que se supera el caudal medio", round(pct_media), "%", f)]
exp += [("1.6", k, round(v, 3), "m³/s", f) for k, v in car.items()]
exp += [("1.6", "Q90 / Q50", round(car["Q90"] / car["Q50"], 2), "—", f)]
if AREA_PASO2_KM2: exp += [("1.6", "Caudal específico medio", round(1000 * Qmed / AREA_PASO2_KM2, 1), "L/s/km²", f)]
EXP = pd.DataFrame([("1",) + x for x in exp], columns=["capitulo", "apartado", "campo", "valor", "unidad", "fuente"])
EXP.to_csv(f"salidas/expediente_P3_{CUENCA}_{SLUG}.csv", index=False, encoding="utf-8-sig")
Qs.to_frame().to_csv(f"salidas/serie_caudal_salida_{CUENCA}_{SLUG}.csv", encoding="utf-8-sig")
import shutil
paquete = shutil.make_archive(f"Practica1_Paso3_{CUENCA}_{SLUG}", "zip", "salidas")
print("✅ Paquete:", paquete)
try:
    from google.colab import files; files.download(paquete)
except ImportError:
    print("Descárgalo desde el panel de archivos de Colab.")
EXP

---
## Para tu capítulo (cinco líneas cada una)
1. **Curva de gasto.** Compara tu h₀ a mano con el del ajuste. ¿Cambió la curva entre el periodo completo y los
   últimos tres años? ¿Qué le pasó al cauce?
2. **Modelo.** ¿Cuánto sesgo tenía GEOGloWS sin corregir en tu estación y cuánto después de corregir? Si tu
   estación está aguas abajo de una represa (La Esperanza, Poza Honda), ¿qué parte de la curva no reproduce el modelo y por qué?
3. **Caudal medio vs. Q50.** ¿Cuál usarías para diseñar un abastecimiento de agua y por qué?
4. **Régimen.** Con Q90 ÷ Q50, ¿tu río es regular o estacional? Relaciónalo con la forma y el relieve del paso 2.